# 08 — Metrics & Evaluation

How to correctly measure classification, regression, and clustering performance.

## Classification Metrics: Basics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = clf.predict(X_test)

accuracy_score(y_test, y_pred)
precision_score(y_test, y_pred, average="macro")   # "binary" for 2-class, "macro"/"weighted"/"micro" for multiclass
recall_score(y_test, y_pred, average="macro")
f1_score(y_test, y_pred, average="macro")


## Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))


## ROC Curve & AUC (binary/multiclass probabilistic evaluation)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay

y_proba = clf.predict_proba(X_test)[:, 1]   # probability of the positive class

auc = roc_auc_score(y_test, y_proba)
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.plot([0, 1], [0, 1], "k--")   # random-chance baseline
plt.show()

# multiclass: pass all class probabilities with multi_class="ovr" or "ovo"
roc_auc_score(y_test, clf.predict_proba(X_test), multi_class="ovr", average="macro")


## Precision-Recall Curve (better than ROC for imbalanced data)

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, PrecisionRecallDisplay

ap = average_precision_score(y_test, y_proba)
PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.show()


## Choosing a Classification Threshold

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# pick threshold that maximizes F1
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1s.argmax()
best_threshold = thresholds[best_idx]

y_pred_custom = (y_proba >= best_threshold).astype(int)


## Regression Metrics

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

y_pred_reg = reg.predict(X_test)

mean_squared_error(y_test, y_pred_reg)
mean_squared_error(y_test, y_pred_reg, squared=False)   # RMSE (older sklearn) — or use root_mean_squared_error
mean_absolute_error(y_test, y_pred_reg)
r2_score(y_test, y_pred_reg)
mean_absolute_percentage_error(y_test, y_pred_reg)


## Clustering Metrics

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# unsupervised (no ground truth labels needed)
silhouette_score(X, cluster_labels)          # higher is better, range [-1, 1]
calinski_harabasz_score(X, cluster_labels)   # higher is better
davies_bouldin_score(X, cluster_labels)       # lower is better

# supervised (when true labels ARE known, to evaluate clustering quality)
adjusted_rand_score(y_true, cluster_labels)
normalized_mutual_info_score(y_true, cluster_labels)


## Custom Scorers (for use in cross_val_score / GridSearchCV)

In [ ]:
from sklearn.metrics import make_scorer, fbeta_score

f2_scorer = make_scorer(fbeta_score, beta=2)   # weigh recall higher than precision
cross_val_score(clf, X_train, y_train, cv=5, scoring=f2_scorer)

# custom function scorer
def custom_metric(y_true, y_pred):
    return (y_true == y_pred).mean()

custom_scorer = make_scorer(custom_metric)


## Common Gotchas
- `average="macro"` treats all classes equally; `average="weighted"` accounts for class imbalance — pick based on whether minority classes matter as much as majority ones.
- Accuracy is misleading on imbalanced datasets — prefer F1, precision-recall AUC, or balanced accuracy.
- `predict_proba` isn't available on all classifiers by default (e.g. `SVC` needs `probability=True`, which also changes CV internally and adds cost).
- ROC-AUC can look good even with poor precision on rare positive classes — use precision-recall curves when positives are rare.